# Engine Dispatch Verification — EURUSD April 2024

Proves NautilusTrader's two-layer bar dispatch model end-to-end:
1. The `BacktestEngine` internal data stream (`engine.data`) holds raw 1-min EXTERNAL bars before `run()`.
2. The `DataEngine` dispatches every cached bar at runtime to subscribed strategies.
3. The `TimeBarAggregator` (one per `INTERNAL@...-EXTERNAL` subscription) consumes those dispatched EXTERNAL bars and emits aggregated bars to `on_bar`.

| CSV | Source | What it represents |
|---|---|---|
| `csv/raw_engine_data_1min_2024_04.csv` | `engine.data` **before** `engine.run()`, filtered to 1-MIN-EXTERNAL | what the engine **holds** pre-dispatch |
| `csv/sniffer_engine_1min_2024_04.csv` | runtime sniffer, `aggregation_source==EXTERNAL` | bars the `DataEngine` **dispatched** |
| `csv/sniffer_strategy_5min_2024_04.csv` | runtime sniffer, `aggregation_source==INTERNAL` | 5-min bars the **aggregator emitted** to `on_bar` |

All three share the same 12 columns: `stream, price_type, bar_type, ts_event_ns, ts_init_ns, open, high, low, close, volume, ts_event, ts_init`.

End-of-notebook assertion (key = `(bar_type, ts_init_ns)`):
```
set(CSV 1) == set(CSV 2)   →   missing=0  extra=0
```

In [ ]:
import csv
import sys
from decimal import Decimal
from pathlib import Path

import pandas as pd


def _find_project_root() -> Path:
    marker = Path("core") / "instrument_factory.py"

    def _walk_up(p: Path) -> Path | None:
        for parent in (p, *p.parents):
            if (parent / marker).is_file():
                return parent
        return None

    found = _walk_up(Path.cwd())
    if found:
        return found
    for var in ("__vsc_ipynb_file__", "__session__", "__file__"):
        nb = globals().get(var)
        if nb:
            found = _walk_up(Path(nb).resolve().parent)
            if found:
                return found
    for child in Path.cwd().iterdir():
        if child.is_dir() and (child / marker).is_file():
            return child
    raise RuntimeError(f"could not find project root (no `{marker}`) from cwd={Path.cwd()}")


PROJECT_ROOT = _find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

CATALOG_PATH = str(PROJECT_ROOT / "catalog")
CSV_DIR = PROJECT_ROOT / "csv"
CSV_DIR.mkdir(exist_ok=True)

INSTRUMENT_ID_STR = "EURUSD.FOREX_MS"
START = "2024-04-01"
END = "2024-04-30"

CSV_RAW_ENGINE      = CSV_DIR / "raw_engine_data_1min_2024_04.csv"
CSV_SNIFFER_ENGINE  = CSV_DIR / "sniffer_engine_1min_2024_04.csv"
CSV_SNIFFER_STRAT   = CSV_DIR / "sniffer_strategy_5min_2024_04.csv"

print(f"project root: {PROJECT_ROOT}")
print(f"catalog     : {CATALOG_PATH}")
print(f"csv dir     : {CSV_DIR}")

project root: C:\Users\HP\OneDrive\Desktop\m-cube_version1
catalog     : C:\Users\HP\OneDrive\Desktop\m-cube_version1\catalog
csv dir     : C:\Users\HP\OneDrive\Desktop\m-cube_version1\csv


In [ ]:
from nautilus_trader.backtest.engine import BacktestEngine
from nautilus_trader.cache.config import CacheConfig
from nautilus_trader.config import BacktestEngineConfig, LoggingConfig, RiskEngineConfig, StrategyConfig
from nautilus_trader.model import TraderId
from nautilus_trader.model.currencies import USD
from nautilus_trader.model.data import Bar, BarType
from nautilus_trader.model.enums import AccountType, AggregationSource, OmsType
from nautilus_trader.model.identifiers import InstrumentId
from nautilus_trader.model.objects import Money
from nautilus_trader.persistence.catalog.parquet import ParquetDataCatalog
from nautilus_trader.trading.strategy import Strategy

from core.instrument_factory import create_instrument

## Load 1-min BID + ASK bars from the catalog

Mirrors `core.backtest_runner._cached_catalog_bars` — same `+1d -1ns` end-of-day expansion so April 30 is fully included.

In [ ]:
BAR_TYPE_1M_BID = f"{INSTRUMENT_ID_STR}-1-MINUTE-BID-EXTERNAL"
BAR_TYPE_1M_ASK = f"{INSTRUMENT_ID_STR}-1-MINUTE-ASK-EXTERNAL"

catalog = ParquetDataCatalog(CATALOG_PATH)
start_ts = pd.Timestamp(START, tz="UTC")
end_ts   = pd.Timestamp(END,   tz="UTC") + pd.Timedelta(days=1) - pd.Timedelta(nanoseconds=1)

bars_bid = catalog.bars(bar_types=[BAR_TYPE_1M_BID], start=start_ts, end=end_ts)
bars_ask = catalog.bars(bar_types=[BAR_TYPE_1M_ASK], start=start_ts, end=end_ts)
all_bars = list(bars_bid) + list(bars_ask)

print(f"BID bars: {len(bars_bid):,}")
print(f"ASK bars: {len(bars_ask):,}")
print(f"total   : {len(all_bars):,}")

BID bars: 31,680
ASK bars: 31,680
total   : 63,360


In [ ]:
instrument = create_instrument("EUR", "USD", venue="FOREX_MS")

engine = BacktestEngine(config=BacktestEngineConfig(
    trader_id=TraderId("DISPATCH-VERIFY-001"),
    logging=LoggingConfig(bypass_logging=True),
    risk_engine=RiskEngineConfig(bypass=True),
    run_analysis=False,
    cache=CacheConfig(bar_capacity=200_000, drop_instruments_on_reset=False),
))
engine.add_venue(
    venue=instrument.id.venue,
    oms_type=OmsType.NETTING,
    account_type=AccountType.MARGIN,
    starting_balances=[Money(1_000_000, USD)],
    base_currency=USD,
    default_leverage=Decimal(1),
)
engine.add_instrument(instrument)
engine.add_data(all_bars)
print("engine built; instrument + venue + bars registered")

engine built; instrument + venue + bars registered


## CSV 1 — raw engine pre-run dump

Reads `engine.data` — the engine's internal data stream, populated by `engine.add_data(...)`. Filtered to `Bar` instances with `aggregation_source==EXTERNAL`.

Note: `engine.cache.bars(...)` does **not** hold the pre-run buffer — the cache is populated by the `DataEngine` at dispatch time during `engine.run()`. The pre-run staging area is `engine.data` (returns a copy of the internal `_data` list).

In [ ]:
CSV_FIELDS = [
    "stream", "price_type", "bar_type",
    "ts_event_ns", "ts_init_ns",
    "open", "high", "low", "close", "volume",
    "ts_event", "ts_init",
]


def bar_to_row(stream: str, bar: Bar) -> dict:
    bt = bar.bar_type
    return {
        "stream":      stream,
        "price_type":  bt.spec.price_type.name,
        "bar_type":    str(bt),
        "ts_event_ns": int(bar.ts_event),
        "ts_init_ns":  int(bar.ts_init),
        "open":        float(bar.open),
        "high":        float(bar.high),
        "low":         float(bar.low),
        "close":       float(bar.close),
        "volume":      float(bar.volume),
        "ts_event":    pd.Timestamp(bar.ts_event, unit="ns", tz="UTC").isoformat(),
        "ts_init":     pd.Timestamp(bar.ts_init,  unit="ns", tz="UTC").isoformat(),
    }


def write_rows(path, rows):
    with open(path, "w", newline="", encoding="utf-8") as fh:
        w = csv.DictWriter(fh, fieldnames=CSV_FIELDS)
        w.writeheader()
        w.writerows(rows)


raw_rows = [
    bar_to_row("raw_engine", d)
    for d in engine.data
    if isinstance(d, Bar) and d.bar_type.aggregation_source == AggregationSource.EXTERNAL
]
write_rows(CSV_RAW_ENGINE, raw_rows)
ext_types_seen = sorted({r["bar_type"] for r in raw_rows})
print(f"EXTERNAL bar_types in engine.data: {ext_types_seen}")
print(f"CSV 1 written: {CSV_RAW_ENGINE.name}  rows={len(raw_rows):,}")

EXTERNAL bar_types in engine.data: ['EURUSD.FOREX_MS-1-MINUTE-ASK-EXTERNAL', 'EURUSD.FOREX_MS-1-MINUTE-BID-EXTERNAL']
CSV 1 written: raw_engine_data_1min_2024_04.csv  rows=63,360


## Sniffer strategy

Subscribes to four BarTypes — two EXTERNAL (1-min BID + ASK, the raw feed) and two INTERNAL composites (5-min BID + ASK, built by `TimeBarAggregator` from the EXTERNAL feed). `on_bar` routes by `bar.bar_type.aggregation_source`; `on_stop` writes the two output CSVs.

In [ ]:
class BarSnifferConfig(StrategyConfig, frozen=True):
    instrument_id: InstrumentId
    bar_types: tuple[BarType, ...]
    csv_engine_path: str
    csv_strategy_path: str


class BarSniffer(Strategy):
    """Records every on_bar; buckets EXTERNAL vs INTERNAL; writes 2 CSVs in on_stop."""

    def __init__(self, config: BarSnifferConfig) -> None:
        super().__init__(config)
        self._engine_rows: list[dict] = []
        self._strategy_rows: list[dict] = []

    def on_start(self) -> None:
        for bt in sorted(self.config.bar_types, key=lambda b: b.aggregation_source.value):
            self.subscribe_bars(bt)

    def on_bar(self, bar: Bar) -> None:
        if bar.bar_type.aggregation_source == AggregationSource.EXTERNAL:
            self._engine_rows.append(bar_to_row("sniffer_engine", bar))
        else:
            self._strategy_rows.append(bar_to_row("sniffer_strategy", bar))

    def on_stop(self) -> None:
        write_rows(self.config.csv_engine_path,   self._engine_rows)
        write_rows(self.config.csv_strategy_path, self._strategy_rows)

## Run

In [ ]:
bt_1m_bid = BarType.from_str(f"{INSTRUMENT_ID_STR}-1-MINUTE-BID-EXTERNAL")
bt_1m_ask = BarType.from_str(f"{INSTRUMENT_ID_STR}-1-MINUTE-ASK-EXTERNAL")
bt_5m_bid_int = BarType.from_str(f"{INSTRUMENT_ID_STR}-5-MINUTE-BID-INTERNAL@1-MINUTE-EXTERNAL")
bt_5m_ask_int = BarType.from_str(f"{INSTRUMENT_ID_STR}-5-MINUTE-ASK-INTERNAL@1-MINUTE-EXTERNAL")

sniffer = BarSniffer(BarSnifferConfig(
    instrument_id=instrument.id,
    bar_types=(bt_1m_bid, bt_1m_ask, bt_5m_bid_int, bt_5m_ask_int),
    csv_engine_path=str(CSV_SNIFFER_ENGINE),
    csv_strategy_path=str(CSV_SNIFFER_STRAT),
))
engine.add_strategy(sniffer)
engine.run()
print(
    f"sniffer captured: engine={len(sniffer._engine_rows):,} "
    f"strategy={len(sniffer._strategy_rows):,}"
)
engine.dispose()

sniffer captured: engine=63,360 strategy=17,280


## Cross-check — `raw_engine == sniffer_engine`

Key is `(bar_type, ts_init_ns)` because BID and ASK at the same source minute share `ts_init`. CSV 3 is informational only.

In [ ]:
df1 = pd.read_csv(CSV_RAW_ENGINE)
df2 = pd.read_csv(CSV_SNIFFER_ENGINE)
df3 = pd.read_csv(CSV_SNIFFER_STRAT)

key = ["bar_type", "ts_init_ns"]
s1 = set(map(tuple, df1[key].itertuples(index=False, name=None)))
s2 = set(map(tuple, df2[key].itertuples(index=False, name=None)))

missing = s1 - s2  # in pre-run cache but never dispatched
extra   = s2 - s1  # dispatched but not present in pre-run cache

print(f"CSV 1 (raw_engine, 1-min EXT):     {len(df1):>7,} rows")
print(f"CSV 2 (sniffer_engine, 1-min EXT): {len(df2):>7,} rows")
print(f"CSV 3 (sniffer_strategy, 5-min):   {len(df3):>7,} rows")
print(f"missing={len(missing)}  extra={len(extra)}")

assert len(missing) == 0, f"engine dropped bars: {list(missing)[:3]}"
assert len(extra)   == 0, f"engine invented bars: {list(extra)[:3]}"

per_side = df3.groupby("price_type").size().to_dict()
print(f"5-MIN INTERNAL per side: {per_side}  (clock-driven; expect ~8,500-8,600)")

CSV 1 (raw_engine, 1-min EXT):      63,360 rows
CSV 2 (sniffer_engine, 1-min EXT):  63,360 rows
CSV 3 (sniffer_strategy, 5-min):    17,280 rows
missing=0  extra=0
5-MIN INTERNAL per side: {'ASK': 8640, 'BID': 8640}  (clock-driven; expect ~8,500-8,600)
